# 22DM015 Final Project — Financial PhraseBank
**Part 2 — Zero-Shot**

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
import pandas as pd

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

SEED = du.SEED
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# load splits
splits = du.load_splits()          
train, val, test = splits['train'], splits['val'], splits['test']

## Part 2c.‍ Zero-Shot Learning with an LLM 

For this part, we chose a deliberately minimal set-up which is a single instruction prompt that asks the model to classify each test sentence as exactly one of `negative`, `neutral`, or `positive`.‍ Note that while we know we can improve this by providing annotator instructions (e.g.‍ using the investor's point of view, which we have learned from Part 1), we wanted to see how a basic prompt performs with a powerful LLM, which we chose to be the latest from Claude (Opus 4.8).‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2c: zero-shot classification of the shared test split (453 sentences) with Claude.
# Needs the ANTHROPIC_API_KEY environment variable (console.anthropic.com -> API Keys).
import os, hashlib, time

import pandas as pd

try:
    import anthropic
    HAVE_SDK = True
except ImportError:           # pip install anthropic
    HAVE_SDK = False

LLM_MODEL = 'claude-opus-4-8'        # exact model id, declared per course rules
CACHE_CSV = f'../.cache/llm_responses_{LLM_MODEL}.csv'
LABELS = ['negative', 'neutral', 'positive']   # index == label id (shared contract)

ZERO_SHOT_PROMPT = (
    'Classify the sentiment of this financial sentence as exactly one of: '
    'negative, neutral, positive. Reply with only that word.\n\nSentence: {text}'
)


def logged(method):
    """Latest TEST row for (LLM_MODEL, method) from the shared scoreboard. The eval module
    keys rows on (model, method, split, n_train_labeled) and no longer tracks person, so
    we match on LLM_MODEL + method here."""
    if not eu.RESULTS_CSV.exists():
        return None
    r = pd.read_csv(eu.RESULTS_CSV)
    r = r[(r['model'] == LLM_MODEL) & (r['method'] == method) & (r['split'] == 'test')]
    return {k: r.iloc[-1][k] for k in eu.METRIC_KEYS} if len(r) else None


def parse_label(reply):
    """Map free-text LLM output to a label id robustly; None if no label word found."""
    hits = [(reply.lower().find(w), i) for i, w in enumerate(LABELS)]
    hits = [(pos, i) for pos, i in hits if pos >= 0]
    return min(hits)[1] if hits else None     # first label word mentioned wins


def _sha1(s):
    return hashlib.sha1(s.encode('utf-8')).hexdigest()


def _flush_cache(new):
    """Merge new replies into the cache CSV atomically (tmp + replace)."""
    add = pd.DataFrame([{'kind': k[0], 'key': k[1], 'raw': v} for k, v in new.items()])
    prev = (pd.read_csv(CACHE_CSV, keep_default_na=False)
            if os.path.exists(CACHE_CSV) else pd.DataFrame())
    merged = (pd.concat([prev, add], ignore_index=True)
              .drop_duplicates(subset=['kind', 'key'], keep='last'))
    os.makedirs(os.path.dirname(CACHE_CSV), exist_ok=True)
    merged.to_csv(CACHE_CSV + '.tmp', index=False)
    os.replace(CACHE_CSV + '.tmp', CACHE_CSV)


def classify_split(kind, df, system=None, prompt=ZERO_SHOT_PROMPT, min_interval=1.4):
    """Classify df['text'] sequentially, paced to stay under the org rate limit.
    Replies are cached under (kind:prompt-fingerprint, sha1(text)); the cache is
    flushed every 50 calls and on failure, so an interrupted run resumes for free."""
    ns = f"{kind}:{_sha1(prompt + chr(0) + (system or ''))[:8]}"   # prompt-aware namespace
    client = anthropic.Anthropic(max_retries=10)   # 429s also back off per retry-after
    cache = {}
    if os.path.exists(CACHE_CSV):
        prev = pd.read_csv(CACHE_CSV, keep_default_na=False)
        cache = {(r['kind'], r['key']): str(r['raw']) for _, r in prev.iterrows()}

    new, raws, last = {}, [], 0.0
    try:
        for i, text in enumerate(df['text']):
            k = (ns, _sha1(text))
            if k in cache:
                raws.append(cache[k])
                continue
            wait = min_interval - (time.monotonic() - last)
            if wait > 0:
                time.sleep(wait)
            last = time.monotonic()
            resp = client.messages.create(
                model=LLM_MODEL, max_tokens=16, system=system or anthropic.NOT_GIVEN,
                messages=[{'role': 'user', 'content': prompt.format(text=text)}],
            )
            raw = next((b.text for b in resp.content if b.type == 'text'), '')
            if raw.strip():               # never cache an empty reply — re-ask next run
                new[k] = raw
            raws.append(raw)
            if len(new) % 50 == 0:
                _flush_cache(new)
                print(f'  {i + 1}/{len(df)} classified ({kind})')
    finally:
        if new:                       # keep progress even if a call ultimately failed
            _flush_cache(new)
            print(f'[cache] {len(new)} new replies saved ({ns})')

    preds = [parse_label(r) for r in raws]
    n_bad = sum(p is None for p in preds)
    if n_bad > 0.05 * len(df):        # neutral-defaulting >5% would inflate accuracy
        raise RuntimeError(f'{n_bad}/{len(df)} replies unparsable — fix the prompt/parsing '
                           'instead of silently defaulting them to the majority class')
    if n_bad:
        print(f'[warn] {n_bad} unparsable replies -> defaulted to neutral (majority class)')
    return [1 if p is None else p for p in preds], n_bad

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
zs_m = logged('zero-shot')
if zs_m is not None:
    print('[cached]')
elif not HAVE_SDK:
    print('[waiting] anthropic SDK not installed — pip install anthropic, then re-run.')
elif not os.environ.get('ANTHROPIC_API_KEY'):
    print('[waiting] ANTHROPIC_API_KEY not set — set the env var and re-run this cell.')
else:
    zs_preds, zs_bad = classify_split('zero-shot', test)
    zs_m = eu.evaluate(test['label'].values, zs_preds)
    eu.log_result(LLM_MODEL, 'zero-shot', 0, zs_m,
                  notes=f'zero-shot on test (n=453); unparsable={zs_bad}')
    print('[done]')

if zs_m is not None:
    print('2c zero-shot TEST:', eu.fmt(zs_m))

### 2c - analysis

Test-split metrics, carrying over the Part 2a/2b rows and adding the 2c zero-shot LLM (`claude-opus-4-8`) result:

| Variant | Accuracy | F1 macro | F1 weighted | F1 negative | F1 neutral | F1 positive |
|---|---|---|---|---|---|---|
| Prior-weighted random (baseline) | 0.4525 | 0.3282 | 0.4540 | 0.1077 | 0.6043 | 0.2727 |
| Rule-based / directional (baseline) | 0.8278 | 0.7883 | 0.8170 | 0.8148 | 0.8762 | 0.6739 |
| 2a full fine-tuning (`32-shot`) | 0.6887 | 0.5538 | 0.7004 | 0.2791 | 0.8701 | 0.5122 |
| 2a frozen linear probe (`32-shot-frozen`) | 0.4459 | 0.3792 | 0.4789 | 0.2833 | 0.6237 | 0.2308 |
| 2b augmented, full (`augmented`) | 0.7991 | 0.7157 | 0.8035 | 0.5263 | 0.8969 | 0.7239 |
| 2b augmented, frozen (`augmented-frozen`) | 0.7064 | 0.5896 | 0.7042 | 0.4286 | 0.8541 | 0.4862 |
| 2c zero-shot LLM (`claude-opus-4-8`) | 0.9823 | 0.9779 | 0.9824 | 0.9683 | 0.9873 | 0.9780 |

The zero-shot LLM gives the highest numbers in the project.‍ `claude-opus-4-8` reaches 0.978 macro-F1 with zero labelled examples, above 32-shot BERT (0.554), augmented BERT (0.716), and even the LLM-generated 2d run.‍ The per-class scores are uniformly strong (negative 0.968, neutral 0.987, positive 0.978), and there is no minority-class collapse which we saw all the previous BERT models.‍

However, we need to consider potential benchmark contamination.‍ Financial PhraseBank is a public dataset and a frontier LLM has very likely seen it, or near-identical text, in pretraining.‍ A high score could measure partly the recall of a memorised benchmark rather than pure generalisation.‍ A clean re-test would require an out-of-distribution split that did not exist at the model's training cut-off.‍

In addition, if this model is put into production, we have some costs and latency problems.‍ Each prediction is a paid API call subject to rate limits and ~1.4 s pacing per request, which costs about 12 minutes for the 453-sentence test pass and recurring spend for any production deployment, against a fine-tuned local BERT that runs offline at near-zero marginal cost.‍ Also, the LLM is a black-box API.‍